In [31]:
import pandas as pd
import numpy as np
import os

In [32]:
df = pd.read_csv("../data/raw/APY.csv")

print("Original shape:", df.shape)

Original shape: (345336, 8)


In [33]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield']


In [34]:
text_columns = ["State", "District", "Crop", "Season"]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

In [35]:
print(df["Crop"].unique())

<StringArray>
[             'Arecanut',             'Arhar/Tur',                'Banana',
          'Black pepper',             'Cashewnut',               'Coconut',
         'Cowpea(Lobia)',          'Dry chillies',                'Ginger',
             'Groundnut',                 'Maize',     'Moong(Green Gram)',
        'Oilseeds total',   'Other Kharif pulses',        'other oilseeds',
     'Rapeseed &Mustard',                  'Rice',               'Sesamum',
             'Sugarcane',             'Sunflower',          'Sweet potato',
               'Tapioca',              'Turmeric',                  'Urad',
                 'Bajra',           'Castor seed',             'Coriander',
          'Cotton(lint)',                'Garlic',                  'Gram',
             'Guar seed',            'Horse-gram',                 'Jowar',
               'Linseed',                'Masoor',                 'Mesta',
            'Niger seed',                 'Onion',    'Other  Rabi pulses'

In [36]:
state_mapping = {
    "THE DADRA AND NAGAR HAVELI": "Dadra and Nagar Haveli",
    "Laddak": "Ladakh",
    "CHANDIGARH": "Chandigarh"
}

df["State"] = df["State"].replace(state_mapping)

In [37]:
print(sorted(df["State"].dropna().unique()))
print("Number of states:", df["State"].nunique())

['Andaman and Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
Number of states: 36


In [38]:
print("Missing Crop before:", df["Crop"].isna().sum())

Missing Crop before: 9


In [39]:
df = df.dropna(subset=["Crop"]).copy()

In [40]:
print("Missing Crop after:", df["Crop"].isna().sum())

Missing Crop after: 0


In [41]:
print("Missing Production:", df["Production"].isna().sum())

Missing Production: 4944


In [42]:
df = df.dropna(subset=["Production"]).copy()

In [43]:
print(df.isnull().sum())

State         0
District      0
Crop          0
Crop_Year     0
Season        0
Area          0
Production    0
Yield         0
dtype: int64


In [44]:
zero_production = df[df["Production"] == 0]

zero_production[
    ["State", "District", "Crop",
     "Crop_Year", "Season",
     "Area", "Production", "Yield"]
].head(20)

,State,District,Crop,Crop_Year,Season,Area,Production,Yield
59,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Arhar/Tur,2015,Rabi,0.50,0.0,0.60
60,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Arhar/Tur,2016,Rabi,6.50,0.0,0.05
63,Andaman and Nicobar Island,SOUTH ANDAMANS,Arhar/Tur,2013,Rabi,0.50,0.0,0.40
64,Andaman and Nicobar Island,SOUTH ANDAMANS,Arhar/Tur,2014,Rabi,1.00,0.0,0.40
65,Andaman and Nicobar Island,SOUTH ANDAMANS,Arhar/Tur,2015,Rabi,0.50,0.0,0.40
124,Andaman and Nicobar Island,NICOBARS,Black pepper,2005,Whole Year,41.00,0.0,0.00
133,Andaman and Nicobar Island,NICOBARS,Black pepper,2017,Rabi,12.40,0.0,0.03
134,Andaman and Nicobar Island,NICOBARS,Black pepper,2018,Rabi,10.41,0.0,0.00
189,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Cashewnut,2017,Rabi,0.80,0.0,0.55
329,Andaman and Nicobar Island,NICOBARS,Ginger,2017,Rabi,0.61,0.0,0.02


In [45]:
print("Zero production rows:", len(zero_production))

Zero production rows: 1465


In [46]:
inconsistent_zero = df[
    (df["Production"] == 0) &
    (df["Yield"] > 0)
]

print(
    "Production = 0 but Yield > 0:",
    len(inconsistent_zero)
)

inconsistent_zero.head(20)

Production = 0 but Yield > 0: 434


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
59,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Arhar/Tur,2015,Rabi,0.50,0.0,0.60
60,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Arhar/Tur,2016,Rabi,6.50,0.0,0.05
63,Andaman and Nicobar Island,SOUTH ANDAMANS,Arhar/Tur,2013,Rabi,0.50,0.0,0.40
64,Andaman and Nicobar Island,SOUTH ANDAMANS,Arhar/Tur,2014,Rabi,1.00,0.0,0.40
65,Andaman and Nicobar Island,SOUTH ANDAMANS,Arhar/Tur,2015,Rabi,0.50,0.0,0.40
133,Andaman and Nicobar Island,NICOBARS,Black pepper,2017,Rabi,12.40,0.0,0.03
189,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Cashewnut,2017,Rabi,0.80,0.0,0.55
329,Andaman and Nicobar Island,NICOBARS,Ginger,2017,Rabi,0.61,0.0,0.02
429,Andaman and Nicobar Island,SOUTH ANDAMANS,Moong(Green Gram),2017,Rabi,0.50,0.0,0.60
443,Andaman and Nicobar Island,NICOBARS,other oilseeds,2003,Whole Year,4.20,0.0,0.07


In [47]:
print("Area <= 0:", (df["Area"] <= 0).sum())

Area <= 0: 0


In [48]:
df[df["Crop_Year"] == 2020].head(20)

,State,District,Crop,Crop_Year,Season,Area,Production,Yield
326054,Uttarakhand,ALMORA,Arhar/Tur,2020,Kharif,216.0,140.0,0.65
326086,Uttarakhand,CHAMOLI,Arhar/Tur,2020,Kharif,227.0,178.0,0.78
326103,Uttarakhand,CHAMPAWAT,Arhar/Tur,2020,Kharif,2.0,1.0,0.50
326124,Uttarakhand,DEHRADUN,Arhar/Tur,2020,Kharif,387.0,516.0,1.33
326155,Uttarakhand,NAINITAL,Arhar/Tur,2020,Kharif,5.0,3.0,0.60
326176,Uttarakhand,PAURI GARHWAL,Arhar/Tur,2020,Kharif,158.0,124.0,0.78
326186,Uttarakhand,PITHORAGARH,Arhar/Tur,2020,Kharif,2.0,1.0,0.50
326207,Uttarakhand,RUDRA PRAYAG,Arhar/Tur,2020,Kharif,158.0,124.0,0.78
326228,Uttarakhand,TEHRI GARHWAL,Arhar/Tur,2020,Kharif,1741.0,2273.0,1.31
326266,Uttarakhand,UTTAR KASHI,Arhar/Tur,2020,Kharif,431.0,632.0,1.47


In [49]:
df[df["Crop_Year"] == 2020]["State"].value_counts()

State
Uttarakhand    319
Name: count, dtype: Int64

In [50]:
df[df["Crop_Year"] == 2020]["Crop"].value_counts().head(30)

Crop
Garlic                   23
Potato                   23
Maize                    17
Rice                     16
Urad                     16
Barley                   13
Rapeseed &Mustard        13
Peas & beans (Pulses)    13
Wheat                    13
Masoor                   13
Onion                    13
Soyabean                 12
Sesamum                  12
Other Cereals            12
Other Kharif pulses      12
Turmeric                 12
Small millets            11
Horse-gram               11
Ragi                     11
Gram                     11
Arhar/Tur                10
other oilseeds            8
Groundnut                 6
Sugarcane                 5
Moong(Green Gram)         5
Other  Rabi pulses        3
Sunflower                 3
Moth                      1
Tobacco                   1
Name: count, dtype: Int64

In [51]:
os.makedirs("../data/processed", exist_ok=True)

df.to_csv(
    "../data/processed/agriculture_clean.csv",
    index=False
)

print("Clean dataset saved!")
print("Shape:", df.shape)

Clean dataset saved!
Shape: (340383, 8)


In [52]:
df["Production"] == 0

0         False
1         False
2         False
3         False
4         False
          ...  
345331    False
345332    False
345333    False
345334    False
345335    False
Name: Production, Length: 340383, dtype: bool

In [57]:
Production == 0 and Yield > 0

NameError: name 'Production' is not defined

In [54]:
df[df["Crop_Year"] == 2020]["State"].value_counts()

State
Uttarakhand    319
Name: count, dtype: Int64

In [55]:
df[df["Crop_Year"] == 2020]["Crop"].value_counts()

Crop
Garlic                   23
Potato                   23
Maize                    17
Rice                     16
Urad                     16
Barley                   13
Rapeseed &Mustard        13
Peas & beans (Pulses)    13
Wheat                    13
Masoor                   13
Onion                    13
Soyabean                 12
Sesamum                  12
Other Cereals            12
Other Kharif pulses      12
Turmeric                 12
Small millets            11
Horse-gram               11
Ragi                     11
Gram                     11
Arhar/Tur                10
other oilseeds            8
Groundnut                 6
Sugarcane                 5
Moong(Green Gram)         5
Other  Rabi pulses        3
Sunflower                 3
Moth                      1
Tobacco                   1
Name: count, dtype: Int64

In [56]:
crop_yield_stats.head(20)

,count,mean,median,min,max
Crop,,,,,
Coconut,2891,8943.243196,8109.890,0.00,43958.33
Cashewnut,1519,7.002916,0.400,0.00,9801.00
Onion,10621,13.245736,11.000,0.00,4070.00
Maize,20335,2.675144,1.950,0.00,1494.00
Sannhamp,2738,1.516644,0.500,0.00,1022.00
Tobacco,3858,2.702519,1.540,0.00,964.80
Sugarcane,10826,56.188318,55.400,0.00,500.49
Potato,10729,13.182370,10.760,0.00,311.02
Cotton(lint),6318,2.163686,1.640,0.00,300.00


In [58]:
print("Number of states:", df["State"].nunique())

print("\nState values:")
for state in sorted(df["State"].dropna().unique()):
    print(repr(state))

Number of states: 36

State values:
'Andaman and Nicobar Island'
'Andhra Pradesh'
'Arunachal Pradesh'
'Assam'
'Bihar'
'Chandigarh'
'Chhattisgarh'
'Dadra and Nagar Haveli'
'Daman and Diu'
'Delhi'
'Goa'
'Gujarat'
'Haryana'
'Himachal Pradesh'
'Jammu and Kashmir'
'Jharkhand'
'Karnataka'
'Kerala'
'Ladakh'
'Madhya Pradesh'
'Maharashtra'
'Manipur'
'Meghalaya'
'Mizoram'
'Nagaland'
'Odisha'
'Puducherry'
'Punjab'
'Rajasthan'
'Sikkim'
'Tamil Nadu'
'Telangana'
'Tripura'
'Uttar Pradesh'
'Uttarakhand'
'West Bengal'


In [59]:
inconsistent_zero = df[
    (df["Production"] == 0) &
    (df["Yield"] > 0)
]

print("Inconsistent records:", len(inconsistent_zero))

inconsistent_zero[
    [
        "State",
        "District",
        "Crop",
        "Crop_Year",
        "Season",
        "Area",
        "Production",
        "Yield"
    ]
].head(30)

Inconsistent records: 434


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
59,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Arhar/Tur,2015,Rabi,0.50,0.0,0.60
60,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Arhar/Tur,2016,Rabi,6.50,0.0,0.05
63,Andaman and Nicobar Island,SOUTH ANDAMANS,Arhar/Tur,2013,Rabi,0.50,0.0,0.40
64,Andaman and Nicobar Island,SOUTH ANDAMANS,Arhar/Tur,2014,Rabi,1.00,0.0,0.40
65,Andaman and Nicobar Island,SOUTH ANDAMANS,Arhar/Tur,2015,Rabi,0.50,0.0,0.40
133,Andaman and Nicobar Island,NICOBARS,Black pepper,2017,Rabi,12.40,0.0,0.03
189,Andaman and Nicobar Island,NORTH AND MIDDLE ANDAMAN,Cashewnut,2017,Rabi,0.80,0.0,0.55
329,Andaman and Nicobar Island,NICOBARS,Ginger,2017,Rabi,0.61,0.0,0.02
429,Andaman and Nicobar Island,SOUTH ANDAMANS,Moong(Green Gram),2017,Rabi,0.50,0.0,0.60
443,Andaman and Nicobar Island,NICOBARS,other oilseeds,2003,Whole Year,4.20,0.0,0.07


In [60]:
inconsistent_zero["Crop"].value_counts()

Crop
Sesamum               102
Urad                   45
Moong(Green Gram)      45
Ragi                   41
Rapeseed &Mustard      21
Linseed                20
Horse-gram             17
Safflower              14
Small millets          13
Sunflower              11
Niger seed              9
Castor seed             8
Moth                    8
other oilseeds          7
Cardamom                7
Tobacco                 7
Groundnut               6
Other Cereals           6
Arhar/Tur               6
Bajra                   4
Wheat                   4
Coriander               4
Gram                    3
Maize                   3
Mesta                   3
Cotton(lint)            2
Ginger                  2
Other  Rabi pulses      2
Turmeric                2
Jowar                   2
Black pepper            2
Masoor                  1
Cashewnut               1
Cowpea(Lobia)           1
Rice                    1
Sugarcane               1
Barley                  1
Sannhamp                1
Dry chi

In [61]:
zero_production = df[df["Production"] == 0]

print("Zero production:", len(zero_production))

print("\nYield statistics:")
print(zero_production["Yield"].describe())

print("\nCrops:")
print(zero_production["Crop"].value_counts().head(20))

Zero production: 1465

Yield statistics:
count    1465.000000
mean        0.056287
std         0.148391
min         0.000000
25%         0.000000
50%         0.000000
75%         0.050000
max         2.360000
Name: Yield, dtype: float64

Crops:
Crop
Sesamum                  203
Moong(Green Gram)        138
Urad                      93
Maize                     68
Moth                      56
Rapeseed &Mustard         46
Ragi                      46
Groundnut                 45
Gram                      44
Sannhamp                  44
Small millets             44
Barley                    39
Linseed                   35
Other  Rabi pulses        34
Cotton(lint)              34
Peas & beans (Pulses)     32
Banana                    29
Horse-gram                29
Masoor                    29
other oilseeds            29
Name: count, dtype: Int64


In [62]:
crop_yield_stats = (
    df.groupby("Crop")["Yield"]
      .agg(
          count="count",
          mean="mean",
          median="median",
          min="min",
          max="max"
      )
      .sort_values("max", ascending=False)
)

crop_yield_stats.head(20)

,count,mean,median,min,max
Crop,,,,,
Coconut,2891,8943.243196,8109.890,0.00,43958.33
Cashewnut,1519,7.002916,0.400,0.00,9801.00
Onion,10621,13.245736,11.000,0.00,4070.00
Maize,20335,2.675144,1.950,0.00,1494.00
Sannhamp,2738,1.516644,0.500,0.00,1022.00
Tobacco,3858,2.702519,1.540,0.00,964.80
Sugarcane,10826,56.188318,55.400,0.00,500.49
Potato,10729,13.182370,10.760,0.00,311.02
Cotton(lint),6318,2.163686,1.640,0.00,300.00
